# Common mistake #1 — chasing component count destroys your traces

> *"My neighbour's pipeline found 600 cells and mine only finds 200 — let me loosen
> `min_corr` / `min_pnr` until I get 600 too."*

This is the single most common way to ruin a CNMF-E extraction. **More components is
not more signal.** Past the point where you've found the real cells, loosening the
detection thresholds does three bad things at once:

1. **It floods the result with junk** — ghost blobs and noise seeds that survive
   initialisation but match no real neuron.
2. **It *splits* real cells** into several overlapping fragments.
3. **It corrupts the temporal traces** — both the deconvolved `C` and the projected
   `C + YrA` — because the demixing step now has to share each cell's fluorescence
   across overlapping footprints, and the leftover residual (`YrA`) soaks up
   neighbours' transients (cross-talk).

We demonstrate all three on a **synthetic recording with known ground truth**, so we
can measure exactly how wrong the traces get. The effect is worst in *dense* fields —
which is also where people are most tempted to crank up the cell count.

> The two trace flavours: `model.C` is the OASIS-deconvolved estimate (clean AR(1)
> shape); `model.C + model.YrA` is the noisy *projected* trace (the data at that
> footprint after subtracting the other cells). On a clean, well-separated extraction
> they agree (`corr ≳ 0.95`). **Their disagreement is a direct readout of cross-talk**,
> so we use `corr(C, C+YrA)` as a ground-truth-free quality knob throughout.

In [1]:
import sys, os
# Find the repo root (the dir containing the `cnmfe` package) walking upward,
# so the notebook runs from anywhere.
_d = os.getcwd()
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "cnmfe")):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d)                       # repo root (cnmfe package)
sys.path.insert(0, os.path.join(_d, "tests"))  # miniscope_simulator

import warnings, contextlib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from miniscope_simulator import make_miniscope_movie
from cnmfe.pipeline import CNMFe, CNMFeParams
from cnmfe.preprocess import correlation_pnr

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110


@contextlib.contextmanager
def quiet():
    "Silence the (verbose) per-fit progress prints during the sweep."
    with open(os.devnull, "w") as fnull, contextlib.redirect_stdout(fnull):
        yield


def pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / d) if d > 0 else 0.0


def spatial_corr_to_truth(A_true, a_est):
    "Spatial correlation of one estimated footprint against every true footprint."
    return A_true.T @ a_est / (np.linalg.norm(A_true, axis=0) * np.linalg.norm(a_est) + 1e-12)

## A dense ground-truth movie

The realistic simulator (`tests/miniscope_simulator.py`) gives us a 1-photon movie with
a circular GRIN aperture, vasculature, drifting background, ghost cells, photobleaching
and shot noise — plus the **true** footprints (`A_true`) and traces (`C_true`). We pack
~110 neurons into a 160×160 FOV so footprints genuinely overlap: this is the regime
where over-extraction bites.

In [ ]:
sim = make_miniscope_movie(
    n_neurons=110, dims=(160, 160), T=1200, fps=20.0,
    sigma_neuron_range=(2.5, 3.5),   # small, closely-packed cells -> overlap
    npil_neuron_coupling=0.10,        # a little neuropil cross-talk, as in real data
    decay_time_ms=180.0,              # GCaMP8m
    seed=3,
)
movie  = sim["movie"]
A_true = sim["A_true"]                # (H*W, K_true) true cores
C_true = sim["C_true"]                # (K_true, T) true traces
K_true = A_true.shape[1]
T, H, W = movie.shape
print(f"movie {movie.shape}, ground-truth cells placed: K_true = {K_true}")

# Correlation image for footprint overlays (computed once).
idx = np.linspace(0, T - 1, 600).astype(int)
cn, _pnr = correlation_pnr(movie[idx].astype(np.float32), sigma=3.0)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(movie.mean(0), cmap="gray"); ax[0].set_title("mean projection")
ax[1].imshow(cn, cmap="magma"); ax[1].set_title("correlation image (cn)")
ct = sim["centers"]
ax[1].scatter(ct[:, 1], ct[:, 0], s=8, facecolors="none", edgecolors="cyan", lw=0.6)
for a in ax: a.axis("off")
fig.suptitle(f"Synthetic dense FOV — {K_true} true neurons (cyan)"); plt.tight_layout()

movie (1200, 160, 160), ground-truth cells placed: K_true = 110


## The mistake: sweep the thresholds from strict to greedy

We run the *same* extraction five times, only loosening `min_corr` / `min_pnr` each
time. Everything else is held fixed. For each run we record, using the ground truth:

- **`K`** — how many components were extracted.
- **`frac_real`** — fraction of components that actually match a true cell
  (spatial corr > 0.6). The rest are ghosts / noise.
- **`true_hit`** — how many of the 110 true cells were found at all.
- **`n_split`** — how many true cells were fragmented into ≥2 components.
- **`selfr`** — mean `corr(C, C+YrA)` across components (the cross-talk readout).

In [ ]:
def assign_to_truth(A_est, A_true, thr=0.6):
    "For each estimated comp: (best true idx or -1, best spatial corr)."
    Ae = np.asarray(A_est.todense())
    best_idx, best_sp = [], []
    for k in range(Ae.shape[1]):
        sp = spatial_corr_to_truth(A_true, Ae[:, k])
        j = int(np.argmax(sp))
        best_idx.append(j if sp[j] > thr else -1)
        best_sp.append(float(sp[j]))
    return np.array(best_idx), np.array(best_sp)


GRID = [(0.85, 14), (0.80, 10), (0.75, 7), (0.70, 5), (0.65, 3.5)]
rows, models = [], []

for mc, mp in GRID:
    params = CNMFeParams(
        sigma=3.0, min_corr=mc, min_pnr=mp, min_pixel=5,
        decay_time_ms=180.0, frame_rate_hz=20.0, n_iter_main=1, n_jobs=-1,
    )
    with quiet():
        model = CNMFe(params).fit(movie, do_motion_correction=False)
    models.append(model)

    K = model.A.shape[1]
    P = model.C + model.YrA
    selfr = np.array([pearson(model.C[k], P[k]) for k in range(K)])
    assign, best_sp = assign_to_truth(model.A, A_true)
    real = best_sp > 0.6
    cnt = Counter(assign[assign >= 0])
    rows.append(dict(
        min_corr=mc, min_pnr=mp, K=K,
        frac_real=float(real.mean()),
        true_hit=len(cnt),
        n_split=sum(1 for v in cnt.values() if v >= 2),
        selfr_mean=float(selfr.mean()),
        selfr_real=float(np.median(selfr[real])) if real.any() else np.nan,
        selfr_junk=float(np.median(selfr[~real])) if (~real).any() else np.nan,
    ))

print(f"{'min_corr':>8} {'min_pnr':>7} {'K':>4} {'frac_real':>9} "
      f"{'true_hit':>8} {'n_split':>7} {'selfr':>6}")
for r in rows:
    print(f"{r['min_corr']:>8} {r['min_pnr']:>7} {r['K']:>4} {r['frac_real']:>9.2f} "
          f"{r['true_hit']:>8} {r['n_split']:>7} {r['selfr_mean']:>6.3f}")

## What you gain vs what you lose

Read the two panels together. **Left:** as the thresholds loosen, `K` (orange)
explodes far past the 110 true cells — but `true_hit` (green) plateaus around ~107.
You are *not* finding more real cells; you've already found them. Everything above the
green line is junk or fragments, and `frac_real` (the share of components that are real)
collapses. **Right:** the cross-talk readout `corr(C, C+YrA)` falls as `K` rises, and
the number of *split* real cells climbs.

In [ ]:
K   = np.array([r["K"] for r in rows])
mpn = np.array([r["min_pnr"] for r in rows])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ax[0].plot(mpn, K, "-o", color="tab:orange", label="K extracted")
ax[0].plot(mpn, [r["true_hit"] for r in rows], "-o", color="tab:green",
           label="true cells found")
ax[0].axhline(K_true, ls="--", color="gray", lw=1, label=f"K_true = {K_true}")
ax[0].set_xlabel("min_pnr  (looser →)"); ax[0].invert_xaxis()
ax[0].set_ylabel("component count"); ax[0].legend(loc="upper left")
ax0b = ax[0].twinx()
ax0b.plot(mpn, [r["frac_real"] for r in rows], "-s", color="tab:red",
          label="frac real")
ax0b.set_ylabel("fraction of comps that are real", color="tab:red")
ax0b.set_ylim(0, 1); ax0b.tick_params(axis="y", labelcolor="tab:red")
ax[0].set_title("loosening thresholds adds junk, not cells")

ax[1].plot(K, [r["selfr_mean"] for r in rows], "-o", color="tab:blue",
           label="mean corr(C, C+YrA)")
ax[1].set_xlabel("K (components extracted)")
ax[1].set_ylabel("corr(C, C+YrA)", color="tab:blue")
ax[1].tick_params(axis="y", labelcolor="tab:blue")
ax1b = ax[1].twinx()
ax1b.plot(K, [r["n_split"] for r in rows], "-^", color="tab:purple")
ax1b.set_ylabel("# true cells split into ≥2 comps", color="tab:purple")
ax1b.tick_params(axis="y", labelcolor="tab:purple")
ax[1].set_title("traces degrade as cells fragment")
plt.tight_layout()

## The smoking gun: a single over-split cell

Aggregate curves are convincing but abstract. Let's open the hood on **one** real
neuron in the greedy run and look at the actual traces.

We pick the true cell that got split into the most components, and plot its
ground-truth trace against each fragment's `C` (deconvolved) and `C + YrA` (projected).
In the strict run the same cell is a single, clean component.

In [ ]:
strict = models[1]   # min_corr=0.80, min_pnr=10  (~right number of cells)
greedy = models[-1]  # min_corr=0.65, min_pnr=3.5 (3x over-extraction)

assign_g, sp_g = assign_to_truth(greedy.A, A_true)
assign_s, sp_s = assign_to_truth(strict.A, A_true)
cnt_g = Counter(assign_g[assign_g >= 0])
cnt_s = Counter(assign_s[assign_s >= 0])

# Pick the most-split true cell in the greedy run that the strict run captured
# as exactly ONE clean component — so the before/after contrast is fair.
cands = [(kt_, n) for kt_, n in cnt_g.items() if n >= 2 and cnt_s.get(kt_, 0) == 1]
kt = max(cands, key=lambda t: t[1])[0] if cands else max(cnt_g, key=cnt_g.get)

frags = [k for k in range(greedy.A.shape[1]) if assign_g[k] == kt]
frags = sorted(frags, key=lambda k: -(greedy.C + greedy.YrA)[k].max())
strict_k = [k for k in range(strict.A.shape[1]) if assign_s[k] == kt]

gt = C_true[kt]
Pg = greedy.C + greedy.YrA
Ps = strict.C + strict.YrA
tt = np.arange(T) / 20.0

fig, axes = plt.subplots(len(frags) + 1, 1, figsize=(11, 1.5 * (len(frags) + 1)),
                         sharex=True)
# strict reference on top
ax = axes[0]
ax.plot(tt, gt / gt.max(), color="k", lw=1.2, label="ground truth")
if strict_k:
    k = strict_k[0]
    ax.plot(tt, Ps[k] / (Ps[k].max() + 1e-9), color="tab:green", lw=1,
            label=f"strict: 1 comp  (C+YrA r={pearson(Ps[k], gt):.2f},  "
                  f"C r={pearson(strict.C[k], gt):.2f})")
ax.set_title(f"True cell {kt}: STRICT run = one clean component"); ax.legend(
    loc="upper right", fontsize=8); ax.set_yticks([])

for i, k in enumerate(frags):
    ax = axes[i + 1]
    ax.plot(tt, gt / gt.max(), color="k", lw=1.0, alpha=0.6)
    p = Pg[k]
    ax.plot(tt, p / (p.max() + 1e-9), color="tab:red", lw=1)
    ax.set_yticks([])
    ax.set_ylabel(f"frag {i+1}", fontsize=8)
    ax.text(0.005, 0.78, f"C+YrA r={pearson(Pg[k], gt):+.2f}   "
            f"C r={pearson(greedy.C[k], gt):+.2f}   selfr={pearson(greedy.C[k], p):+.2f}",
            transform=ax.transAxes, fontsize=8,
            color="darkred", va="top")
axes[1].set_title(f"GREEDY run: same cell shattered into {len(frags)} fragments "
                  f"(red), each a corrupted copy")
axes[-1].set_xlabel("time (s)"); plt.tight_layout()
print(f"True cell {kt}: strict -> {len(strict_k)} comp, greedy -> {len(frags)} comps")
print("Fragment deconvolved-C correlations with truth:",
      [round(pearson(greedy.C[k], gt), 2) for k in frags])

Each fragment's deconvolved `C` correlation with truth is **markedly worse than the
single clean component the strict run extracted for the same cell** (top panel). When
one neuron's fluorescence is divided across overlapping footprints, the demixing can't
attribute a transient to the "right" fragment, so no fragment carries the whole
trace — *the very signal you would plot and analyse is broken apart.* Depending on how
the split falls, individual fragment correlations can drop to near zero or even go
**negative**. The projected `C + YrA` degrades more gently only because it still
contains the shared raw fluorescence (plus its neighbours' cross-talk) — which is
exactly why `corr(C, C+YrA)` collapses for these components.

## Footprints: strict vs greedy on the correlation image

The greedy run litters the FOV with overlapping and off-cell footprints; the strict run
keeps one tight footprint per real cell.

In [ ]:
def overlay(ax, model, title):
    ax.imshow(cn, cmap="gray")
    Ae = np.asarray(model.A.todense())
    for k in range(Ae.shape[1]):
        a = Ae[:, k].reshape(H, W)
        if a.max() <= 0:
            continue
        ax.contour(a, levels=[0.3 * a.max()], colors=["tab:red"], linewidths=0.5)
    ax.set_title(f"{title}  (K={Ae.shape[1]})"); ax.axis("off")

fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
overlay(ax[0], strict, "strict  (min_corr=0.80, min_pnr=10)")
overlay(ax[1], greedy, "greedy  (min_corr=0.65, min_pnr=3.5)")
plt.tight_layout()

## The lesson

**Don't tune for component count. Tune for trace quality.**

- There is a **density ↔ purity tradeoff**. Once you've found the real cells, every
  extra component you squeeze out by loosening thresholds is junk or a fragment, and it
  *degrades the traces of the cells you already had* through cross-talk.
- `corr(C, C+YrA)` is a **ground-truth-free knob for picking thresholds**: when it
  starts falling as you loosen, you've gone too far. (Here it peaks near the true cell
  count and drops thereafter.)
- In a dense extraction, prefer **`model.C`** (the demixed estimate) as the per-cell
  signal — but note that *over-splitting destroys `C` too*, so the real fix is not to
  over-extract in the first place.
- The auto-evaluation (`model.accepted_mask`) already flags many of the junk
  components by SNR — **use it** instead of trusting raw `K`.

**This is not just a synthetic artefact.** On a real 180×180 dense miniscope cutout,
loosening thresholds took the extraction from K=221 (mean `corr(C, C+YrA)` 0.88,
top-30-amplitude cells 0.77) to K=722 (0.74 / **0.45**) — the strong cells' traces
degraded too. If you genuinely need both high `K` *and* clean traces, sharpen the
demixing instead of loosening detection: `n_iter_main ≥ 2`, tighter footprints
(`spatial_max_thr ↑`, `spatial_circular_max_dist_factor ↓`). On that real cutout that
recipe recovered strong-cell `corr(C, C+YrA)` from 0.48 → 0.77 at K≈600.

See the *"Two trace flavours"* and *"Real-recording tuning"* notes in `CLAUDE.md` for
the full field evidence.